# 01. FinTech Data Exploration & Processing

## Objective
This notebook performs a comprehensive exploration and preprocessing of multiple FinTech datasets.  
The main goals are to understand data distributions, detect anomalies, engineer meaningful features,  
merge datasets, and produce a single, fully-processed dataset ready for machine learning fraud detection.

## Dataset Overview
The datasets we are working with include:

- `transactions.csv`: Historical financial transactions  
- `user_profiles.csv`: User demographic and account profile information  
- `device_fingerprints.csv`: Device information associated with users  
- `ip_reputation.csv`: IP address risk levels and categories  
- `merchant_risk.csv`: Merchant-level risk and chargeback information  
- `login_history.csv`: Login events (successful and failed)  
- `fraud_labels.csv`: Ground truth labels for fraudulent transactions  
- `fraud_rules.csv`: Business rules for fraud flagging  
- `cards.csv`: Card metadata (issuer, status, limit)  
- `merchants.csv`: Merchant details (name, category, MCC, country)  

#### Step 1: Import Required Libraries
We import pandas for tabular data, json for handling JSON streams, and Path for clean file path handling.

In [31]:
# Libraries
import pandas as pd
import json
from pathlib import Path

# Ignore warnings for clean notebook output
import warnings
warnings.filterwarnings('ignore')

print(" Libraries imported successfully")


 Libraries imported successfully


#### Step 2: Define File Paths
- `BASE_FIN` → folder containing raw FinTech datasets  
- `OUTPUT` → folder where we save the processed dataset  
We also ensure the output directory exists.


In [32]:
# STEP 1: Define Directory Paths
BASE_FIN = Path(r"C:\Users\User\Downloads\FinSecAI\data\raw\fintech")
OUTPUT = Path(r"C:\Users\User\Downloads\FinSecAI\data\processed\fintech")

# Create processed folder if it doesn't exist
OUTPUT.mkdir(parents=True, exist_ok=True)

print("Setup complete. Raw and processed directories are ready.")


Setup complete. Raw and processed directories are ready.


#### Step 3: Load All Raw Files
We load every dataset individually to keep source integrity separated.
Each file contains a different dimension of FinTech behavior (transactions, identities, risks, reputation).


In [33]:
 
# Replace with your actual path

transactions = pd.read_csv(BASE_FIN / "transactions.csv")          # Core transaction-level data
stream = pd.read_json(BASE_FIN / "transaction_stream.json")        # Real-time transaction event stream
profiles = pd.read_csv(BASE_FIN / "user_profiles.csv")             # User identity + demographic attributes
devices = pd.read_csv(BASE_FIN / "device_fingerprints.csv")        # Device/browser fingerprint info
ip_rep = pd.read_csv(BASE_FIN / "ip_reputation.csv")               # IP risk scores
merchant_risk = pd.read_csv(BASE_FIN / "merchant_risk.csv")        # Merchant ID + risk scores
login_history = pd.read_csv(BASE_FIN / "login_history.csv")        # Past login attempts, timestamps
fraud_labels = pd.read_csv(BASE_FIN / "fraud_labels.csv")          # Ground-truth fraud labels
fraud_rules = pd.read_csv(BASE_FIN / "fraud_rules.csv")            # Fraud heuristic rule outputs

print("All FinTech raw datasets loaded successfully!")



All FinTech raw datasets loaded successfully!


##### Step 4: Inspect Raw Datasets
We print:
- Name of each dataset  
- Shape (rows × columns)  
- First 5 rows  

This helps confirm:
- Files loaded correctly  
- Schema looks expected  
- Keys we will merge on actually exist  


In [34]:
# Inspect datasets
datasets = {
    "Transactions": transactions,
    "Stream": stream,
    "Profiles": profiles,
    "Devices": devices,
    "IP Reputation": ip_rep,
    "Merchant Risk": merchant_risk,
    "Login History": login_history,
    "Fraud Labels": fraud_labels,
    "Fraud Rules": fraud_rules,
}

for name, df in datasets.items():
    print(f"\n{name}: {df.shape}")
    display(df.head())


print("Data inspection completed for all datasets.")



Transactions: (20, 7)


,transaction_id,user_id,amount,currency,merchant_id,timestamp,channel
0,T1,U9,1971.93,NGN,M5,2025-01-01 00:00:00,WEB
1,T2,U1,648.77,NGN,M7,2025-01-01 01:00:00,POS
2,T3,U4,1678.92,NGN,M2,2025-01-01 02:00:00,MOBILE
3,T4,U2,1853.02,NGN,M4,2025-01-01 03:00:00,POS
4,T5,U10,22.42,NGN,M3,2025-01-01 04:00:00,ATM



Stream: (15, 7)


,event_id,user_id,amount,merchant,device_id,ip,timestamp
0,E1,U8,451.45,M7,D10,102.89.184.51,2025-11-17 11:13:46.227062
1,E2,U5,110.73,M6,D4,102.89.35.174,2025-11-17 11:13:46.227117
2,E3,U10,1414.08,M4,D3,102.89.33.238,2025-11-17 11:13:46.227151
3,E4,U7,100.85,M7,D9,102.89.51.251,2025-11-17 11:13:46.227197
4,E5,U4,1028.76,M4,D2,102.89.227.106,2025-11-17 11:13:46.227246



Profiles: (10, 5)


...


Fraud Labels: (20, 2)


,transaction_id,label
0,T1,0
1,T2,1
2,T3,1
3,T4,0
4,T5,0



Fraud Rules: (10, 3)


,rule_id,description,severity
0,R1,Block transactions above 1M NGN,medium
1,R2,Flag mismatched device fingerprint,medium
2,R3,Flag IPs with high risk level,high
3,R4,Block first-time international transfers,high
4,R5,Flag high-velocity transactions,high


Data inspection completed for all datasets.


#### Step 5: Normalize Column Names
We convert all column names to lowercase to prevent merge mismatches due to case differences.
Example: `User_ID` vs `user_id`.


In [35]:
for df in datasets.values():
    df.columns = df.columns.str.lower()


#### Step 6: Merge Datasets Using Known Keys  
We create a single unified dataset `fin_df`.

##### Merging Strategy:
1. **User-level enrichment**  
   `transactions × profiles` (via `user_id`)

2. **Device fingerprint enrichment**  
   `transactions × devices` (via `device_id`)

3. **Network/IP risk**  
   `transactions × ip_reputation` (via `ip_address`)

4. **Merchant risk scores**  
   `transactions × merchant_risk` (via `merchant_id`)

5. **Behavioral features**  
   Aggregate login history to count per-user login attempts.

6. **Label joining**  
   Add fraud labels (supervised learning target)

7. **Rule-based fraud indicators**  
   Append fraud_rules alongside ML-ready features  


In [38]:
import pandas as pd

# Step 0 — Prepare Stream events to match Transactions schema
stream_df = stream.copy()
stream_df.rename(columns={'event_id':'transaction_id','merchant':'merchant_id'}, inplace=True)
stream_df['currency'] = 'NGN'
stream_df['channel'] = 'STREAM'

# Step 1 — Concatenate Transactions and Stream
combined_df = pd.concat([transactions, stream_df[['transaction_id','user_id','amount','currency','merchant_id','timestamp','channel']]], ignore_index=True)

# Step 2 — Merge with Profiles
combined_df = combined_df.merge(profiles, on='user_id', how='left')

# Step 3 — Attach device_id (prefer Stream, fill with Login History)
login_df = login_history[['user_id','device_id']].rename(columns={'device_id':'device_id_login'})
combined_df = combined_df.merge(login_df, on='user_id', how='left')
combined_df['device_id'] = combined_df.get('device_id')  # keep Stream device if exists
combined_df['device_id'] = combined_df['device_id'].combine_first(combined_df['device_id_login'])
combined_df.drop(columns=['device_id_login'], inplace=True)

# Step 4 — Merge Device Metadata
combined_df = combined_df.merge(devices, on='device_id', how='left')

# Step 5 — Merge IP Reputation
combined_df = combined_df.merge(ip_rep.rename(columns={'ip_address':'last_seen_ip'}), on='last_seen_ip', how='left')

# Step 6 — Merge Merchant Risk
combined_df = combined_df.merge(merchant_risk, on='merchant_id', how='left')

# Step 7 — Merge Fraud Labels (only for original transactions)
combined_df = combined_df.merge(fraud_labels, on='transaction_id', how='left')

# Step 8 — Save final CSV
combined_df.to_csv('processed_fintech.csv', index=False)
print("✅ processed_fintech.csv saved successfully!")


✅ processed_fintech.csv saved successfully!


>After these cleaning steps, all datasets are standardized, consistent, and ready for further analysis or merging into a single `processed_fintech.csv dataset`.